# Heart Disease Prediction Using Machine Learning
# Project Background:

# Heart disease is a major health concern worldwide. Machine learning provides methods for analyzing patient data and adentifying patterns that may be
# associated with present of heart disease.

# This project develops and evaluates several supervised machine learning classification algorithms to predict the presence of heart disease using 
# Scikit-learn pipelines and column tranformers to ensure that preprocessing is performed correctly without data leakage.

# The final model will be evaluated on an unseen test dataset and saved for potential future use.

# This project is intended for educational purpose and machine learning development purooses and is not intended to provide medical diagnosis or 
# clinical recommendations.


# The Problem:
# The objective of this project is to develop a binary classification model capable of predicting whether a patient has heart disease based on 
# available patient characteristics.

# Several machine learning algorithms will be trained and evaluated. Their baseline performance will first be established using cross-validation.
# Hyperparameter tuning will be tuned to investigate whether the performance of the models can be improved. The models will be compared using multiple 
# evaluation metrics, including accuracy, precision, recall, F1-score, and ROC-AUC.

# The final model will be selected based on its cross-validation performance and subsequently evaluated on an unseen test dataset.
     

# PROJECT OBJECTIVES
# The main objectives of this project are to:

# 1.
#     Understand and prepare the heart disease dataset for machine learning.
# 2.
#     Perform exploratory data analysis to indentify important patterns and relationships.
# 3.
#     Indentify numerical and caterigorical features.
# 4.
#     Develop appropriate preprocessing pipelines for different machine learning algorithms.
# 5.
#     Establish baseline performance for several classification algorithms.
# 6.
#     Use cross-validation to obtain more reliable estimates of the model performance.
    
# 7.
#     Perform hyperparameter tuning on the candidate models.
# 8.
#     Compare baseline and tuned model perfomance.
# 9.
#     Select the best-performing final model using an appropriate evaluation metric.
# 10.
#     Evaluate the final model on an unseen test dataset.
# 11.
#     Interpret the final model's performance using appropriate evaluation metrics and visualizations.
# 12.
#     Save the trained machine learning models for future use.
    
     
        

# importing libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.model_selection import (train_test_split, GridSearchCV, cross_validate)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (StandardScaler,OneHotEncoder)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
RandomForestClassifier,
AdaBoostClassifier,
GradientBoostingClassifier,
VotingClassifier
)

from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import(
accuracy_score,
precision_score,
recall_score,
f1_score,
roc_auc_score,
classification_report,
ConfusionMatrixDisplay,
RocCurveDisplay
)

import joblib

# Data Cleaning

In [ ]:
df = pd.read_csv("heart.csv")
df.head()

In [ ]:
print(df.shape)

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.nunique()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated(keep=False)]

In [ ]:
df = df.drop_duplicates().copy()
df.shape

In [ ]:
df.duplicated().sum()

# Exploratory Data Analysis(EDA)

# EDA is performed to understand the distribution of the variables, examine the target variable , indentify potential relationships between
# predictors and the target, and detect unsual observations before model development.

In [ ]:
df.describe().T 

In [ ]:
df["target"].value_counts()

In [ ]:
df["target"].value_counts(normalize = True)*100


In [ ]:
sns.countplot(data= df, x="target")

plt.title("Distribution of Heart Disease Target")
plt.xlabel("Target")
plt.ylabel("Count")
plt.show()

In [ ]:
numerical_features = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak"
]

for column in numerical_features:
    plt.figure(figsize = (7, 4))
    sns.histplot(data=df, x=column, kde = True)
    plt.title(f"Distribution of {column}")
    plt.show()

# Numerical_features vs target

In [ ]:

for column in numerical_features:
    plt.figure(figsize = (7, 4))
    sns.boxplot(data=df, x = "target", y = column)
    plt.title(f"{column} by Heart Disease Status")
    plt.show()

In [ ]:
categorical_features = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal"
]

for column in categorical_features:
    plt.figure(figsize=(7, 4))
    sns.countplot(data=df, x = column, hue = "target")
    plt.title(f"{column} by Heart Disease Status")
    plt.show()

In [ ]:
plt.figure(figsize=(18, 8))

sns.heatmap(
    df.corr(numeric_only = True),
    annot = True,
    cmap = "coolwarm",
    fmt="2f"
)
plt.title("Correlation Matrix")
plt.show()

# Feature and Target Definition:

# target represents predicted variable
# All remaining variables are used as predictor features

In [ ]:
X = df.drop(columns = "target")
y = df["target"]
X.shape, y.shape

# Train-Test-Split

In [ ]:
X_train, X_test, y_train ,y_test = train_test_split(
    X,
    y,
    random_state = 42,
    test_size = .2,
    stratify = y
)

In [ ]:
numerical_features = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak"
]
categorical_features = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal"
]
if len(numerical_features) + len(categorical_features) == 13:
    print("proceed")

# Building  the preprocessing Pipelines

In [ ]:
numeric_preprocessor=Pipeline([
    ("imputer", SimpleImputer(strategy = "median")),
    ("scaler", StandardScaler())
])

In [ ]:
catagorical_preprocessor=Pipeline([
    ("imputer", SimpleImputer(strategy = "most_frequent")),
    ("OneHot",OneHotEncoder(handle_unknown = "ignore"))
])

In [ ]:
preprocessor = ColumnTransformer([
    ("num", numeric_preprocessor,
    numerical_features),

    ("cat", catagorical_preprocessor,
    categorical_features)
])

In [ ]:
preprocessor_scaled = preprocessor

In [ ]:
numeric_preprocessor_tree = Pipeline ([
    ("imputer", SimpleImputer(strategy = "median")),
])

preprocessor_tree = ColumnTransformer([
    ("num", numeric_preprocessor_tree,
     numerical_features),
    ("cat",catagorical_preprocessor,
    categorical_features)
])

In [ ]:
models = {
    "Logistic Regression":
    LogisticRegression(max_iter=1000),
    "KNN":
    KNeighborsClassifier(),
    "SVM":
    SVC(probability= True),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "AdaBoost":AdaBoostClassifier(random_state = 42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "Naive Bayes": GaussianNB()
}

In [ ]:
batch_1 = {
    name: models[name]
    for name in ["Logistic Regression", "KNN", "SVM"]
}

batch_2 = {
    name: models[name]
    for name in ["Decision Tree", "AdaBoost"]
}

batch_3 = {
    name: models[name]
    for name in ["Gradient Boosting", "Naive Bayes"]
}

# Baseline Model Evaluation

# Five-fold-cross-valiadation is used to evaluate the baseline models.Models are evaluated in small batches to reduce computaional and memory requirements.

In [ ]:
def evaluate_models(model_dict, preprocessor,X_train, y_train):
    results = []

    for name, model in model_dict.items():

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        scores = cross_validate(
            pipeline,
            X_train,
            y_train,
            cv = 5,
            scoring = {
                "accuracy": "accuracy",
                "precision": "precision",
                "recall": "recall",
                "f1": "f1",
                "roc_auc": "roc_auc"
            },
            n_jobs = 1,
            error_score = "raise"
        )

        results.append({
            "Model": name,
            "Accuracy": scores["test_accuracy"].mean(),
            "Precision": scores["test_precision"].mean(),
            "Recall": scores["test_recall"].mean(),
            "F1": scores["test_f1"].mean(),
            "ROC_AUC": scores["test_roc_auc"].mean()
        })
    return pd.DataFrame(results)


In [ ]:
results_batch_1 = evaluate_models(
    batch_1,
    preprocessor,
    X_train,
    y_train,
)

In [ ]:
results_batch_1 

In [ ]:
import gc
gc.collect()

In [ ]:
results_batch_2 = evaluate_models(
    batch_2,
    preprocessor_tree,
    X_train,
    y_train,
)

In [ ]:
results_batch_2

In [ ]:
gc.collect()

In [ ]:
categorical_preprocessor_dense = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    ))
])

preprocessor_nb = ColumnTransformer([
    ('num', numeric_preprocessor, numerical_features),
    ('cat', categorical_preprocessor_dense, categorical_features)
])

In [ ]:
results_batch_3 = evaluate_models(
    batch_3,
    preprocessor_nb,
    X_train,
    y_train
)

In [ ]:
results_batch_3

In [ ]:
gc.collect()

In [ ]:
batch_4 = {
    "Random Forest": RandomForestClassifier(
        random_state=42,
        n_jobs=1
    )
}

In [ ]:
results_batch_4 = evaluate_models(
    batch_4,
    preprocessor_tree,
    X_train,
    y_train
)

In [ ]:
results_batch_4

In [ ]:
gc.collect()

# Merge all baseline results

In [ ]:
baseline_results = pd.concat(
    [
        results_batch_1,
        results_batch_2,
        results_batch_3,
        results_batch_4
    ],
    ignore_index=True
)

In [ ]:
baseline_results = baseline_results.sort_values(
    by='ROC_AUC',
    ascending=False
).reset_index(drop=True)

In [ ]:
baseline_results


In [ ]:
baseline_results.to_csv(
    'baseline_results.csv',
    index=False
)

# Voting model

In [ ]:
logreg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

svm_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', SVC(probability=True))
])

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor_tree),
    ('model', RandomForestClassifier(
        random_state=42,
        n_jobs=1
    ))
])

In [ ]:
voting_model = VotingClassifier(
    estimators=[
        ('logreg', logreg_pipeline),
        ('svm', svm_pipeline),
        ('rf', rf_pipeline)
    ],
    voting='soft'
)

In [ ]:
voting_scores = cross_validate(
    voting_model,
    X_train,
    y_train,
    cv=5,
    scoring={
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1',
        'roc_auc': 'roc_auc'
    },
    n_jobs=1
)

In [ ]:
voting_results = pd.DataFrame([{
    'Model': 'Voting Classifier',
    'Accuracy': voting_scores['test_accuracy'].mean(),
    'Precision': voting_scores['test_precision'].mean(),
    'Recall': voting_scores['test_recall'].mean(),
    'F1': voting_scores['test_f1'].mean(),
    'ROC_AUC': voting_scores['test_roc_auc'].mean()
}])

voting_results

In [ ]:
baseline_with_voting = pd.concat(
    [baseline_results, voting_results],
    ignore_index=True
)

baseline_with_voting = baseline_with_voting.sort_values(
    by='ROC_AUC',
    ascending=False
).reset_index(drop=True)

baseline_with_voting

# Tuning batches

# All parameters

In [ ]:
param_grids = {
    'Logistic Regression': {
        'model__C': [0.01, 0.1, 1, 10],
        'model__solver': ['liblinear']
    },

    'KNN': {
        'model__n_neighbors': [3, 5, 7, 9, 11],
        'model__weights': ['uniform', 'distance'],
        'model__p': [1, 2]
    },

    'SVM': {
        'model__C': [0.1, 1, 10],
        'model__kernel': ['linear', 'rbf'],
        'model__gamma': ['scale', 'auto']
    },

    'Decision Tree': {
        'model__max_depth': [None, 3, 5, 7, 10],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4]
    },

    'AdaBoost': {
        'model__n_estimators': [50, 100, 200],
        'model__learning_rate': [0.01, 0.1, 1.0]
    },

    'Gradient Boosting': {
        'model__n_estimators': [50, 100, 200],
        'model__learning_rate': [0.01, 0.1, 0.2],
        'model__max_depth': [2, 3, 5]
    },

    'Random Forest': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [None, 5, 10],
        'model__min_samples_split': [2, 5],
        'model__min_samples_leaf': [1, 2]
    }
}

In [ ]:
tuning_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(probability=True),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(
        random_state=42,
        n_jobs=1
    )
}

In [ ]:
tuning_preprocessors = {
    'Logistic Regression': preprocessor,
    'KNN': preprocessor,
    'SVM': preprocessor,

    'Decision Tree': preprocessor_tree,
    'AdaBoost': preprocessor_tree,
    'Gradient Boosting': preprocessor_tree,
    'Random Forest': preprocessor_tree
}

In [ ]:
def tune_models(
    models,
    param_grids,
    preprocessors,
    X_train,
    y_train
):

    results = []
    best_estimators ={}

    for name, model in models.items():

        pipeline = Pipeline([
            ('preprocessor', preprocessors[name]),
            ('model', model)
        ])

        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grids[name],
            cv=5,
            scoring='roc_auc',
            n_jobs=1
        )

        grid.fit(X_train, y_train)

        results.append({
            'Model': name,
            'Best_ROC_AUC': grid.best_score_,
            'Best_Params': grid.best_params_
        })
        best_estimators[name] =  grid.best_estimator_
       
          

    return pd.DataFrame(results), best_estimators

In [ ]:
tuned_results ,  best_estimators = tune_models(
    tuning_models,
    param_grids,
    tuning_preprocessors,
    X_train,
    y_train
)

In [ ]:
tuned_results = tuned_results.sort_values(
    by='Best_ROC_AUC',
    ascending=False
).reset_index(drop=True)

tuned_results

In [ ]:
baseline_auc = baseline_results[
    ['Model', 'ROC_AUC']
].copy()

baseline_auc = baseline_auc.rename(
    columns={'ROC_AUC': 'Baseline_ROC_AUC'}
)

In [ ]:
tuned_auc = tuned_results[
    ['Model', 'Best_ROC_AUC']
].copy()

tuned_auc = tuned_auc.rename(
    columns={'Best_ROC_AUC': 'Tuned_ROC_AUC'}
)

In [ ]:
baseline_vs_tuned = baseline_auc.merge(
    tuned_auc,
    on='Model',
    how='left'
)

In [ ]:
baseline_vs_tuned['Improvement'] = (
    baseline_vs_tuned['Tuned_ROC_AUC']
    - baseline_vs_tuned['Baseline_ROC_AUC']
)

In [ ]:
baseline_vs_tuned = baseline_vs_tuned.sort_values(
    by='Tuned_ROC_AUC',
    ascending=False
).reset_index(drop=True)

In [ ]:
baseline_vs_tuned

In [ ]:
final_results = []

for name, model in best_estimators.items():

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    final_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC_AUC': roc_auc_score(y_test, y_prob)
    })

final_results = pd.DataFrame(final_results)    

In [ ]:
final_results.sort_values(
    'ROC_AUC',
    ascending=False
).reset_index(drop=True)

In [ ]:
final_model = best_estimators['Logistic Regression']

In [ ]:
y_pred = final_model.predict(X_test)
y_prob = final_model.predict_proba(X_test)[:, 1]

In [ ]:
final_results = pd.DataFrame([{
    'Model': 'Final Logistic Regression',
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1': f1_score(y_test, y_pred),
    'ROC_AUC': roc_auc_score(y_test, y_prob)
}])

final_results

In [ ]:
joblib.dump(
    final_model,
    'heart_disease_final_model.pkl'
)

In [ ]:
final_comparison = final_results.copy()

final_comparison

In [ ]:
final_comparison = final_comparison.sort_values(
    by='ROC_AUC',
    ascending=False
).reset_index(drop=True)

final_comparison

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm
)

disp.plot()
plt.title('Confusion Matrix - Final Logistic Regression')
plt.show()

In [ ]:
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_predictions(
    y_test,
    y_prob
)

plt.title('ROC Curve - Final Logistic Regression')
plt.show()

In [ ]:
plt.savefig(
    'final_logistic_regression_roc_curve.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

In [ ]:
final_comparison.to_csv(
    'final_model_results.csv',
    index=False
)

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred
    )
)

# The final Logistic Regression model achieved an accuracy of 85% on the unseen test set. For the positive class, the model achieved a precision of 83%, recall of 91%, and F1-score of 87%. The high recall indicates that the model was able to identify the majority of positive cases in the test data. For the negative class, the model achieved a precision of 88% and recall of 79%. Overall, the results indicate that the final model provides a good balance between identifying positive cases and avoiding false positive predictions.

# The project successfully developed an end-to-end machine-learning classification system for predicting heart disease. Several algorithms were trained and evaluated using cross-validation, followed by hyperparameter tuning of the main candidate models. Hyperparameter tuning improved the cross-validation performance of all tuned algorithms. Logistic Regression achieved the highest tuned cross-validation ROC-AUC of 0.9187 and was selected as the final model based on its overall performance. On the previously unseen test set, the final model achieved 85.25% accuracy, 83.33% precision, 90.91% recall, 86.96% F1-score, and 89.39% ROC-AUC. The results demonstrate that the model can effectively distinguish between the two classes in this dataset, although the relatively small dataset means that further validation on larger and more diverse data would be necessary before considering real-world or clinical deployment.